# PF2e model prediction

In [1]:
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import joblib

## Opening and pre processing of the data

In [2]:

with open('spells.json', 'r', encoding='utf-8') as file:
    raw_data = json.load(file)

if isinstance(raw_data, dict):
    keys = list(raw_data.keys())
    spell_list = raw_data[keys[0]] 
else:
    spell_list = raw_data


parsed_data = []
for spell in spell_list:
    desc = spell.get('description', spell.get('text', ''))

    tags = spell.get('traits', [])
    if not isinstance(tags, list):
        tags = [tags] if pd.notna(tags) else []
        
    school = spell.get('school', '')
    if school:
        tags.append(school)
        
    if desc and tags:
        parsed_data.append({'text': desc, 'tags': tags})

df = pd.DataFrame(parsed_data)
print(f"amount of spells: {len(df)}")

amount of spells: 1403


## Training of the KNC model based on the key words

In [3]:

X = df['text'].str.replace('<[^<]+?>', '', regex=True) 
excluded_tags = {'arcane', 'occult', 'primal', 'divine'}
y = df['tags'].apply(
    lambda tags: [tag for tag in (tags or []) if tag.casefold() not in excluded_tags]
)

vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)
mlb = MultiLabelBinarizer()

X_encoded = vectorizer.fit_transform(X)
y_encoded = mlb.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42
)

print(f"Amount of features: {X_encoded.shape[1]}")
print(f"Amount of unique tags: {len(mlb.classes_)}")

Amount of features: 3000
Amount of unique tags: 103


In [4]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

predictions = knn.predict(X_test)

f1_micro = f1_score(y_test, predictions, average='micro')
accuracy = accuracy_score(y_test, predictions)

print(f"Baseline Accuracy (amount of correct predictions): {accuracy:.2f}")
print(f"Baseline F1-Score (micro): {f1_micro:.2f}")

Baseline Accuracy (amount of correct predictions): 0.07
Baseline F1-Score (micro): 0.38


In [5]:
# test
custom_spell = [
    "A massive ball of superheated plasma explodes, dealing fire damage to anyone in the area."
]
custom_vec = vectorizer.transform(custom_spell)
custom_pred = knn.predict(custom_vec)

print("\ntest spell", custom_spell[0])
print("tags predicted:", mlb.inverse_transform(custom_pred))

#joblib.dump(knn, 'knn_model.joblib')
#joblib.dump(vectorizer, 'vectorizer.joblib')
#joblib.dump(mlb, 'mlb.joblib')

#print("\n")


test spell A massive ball of superheated plasma explodes, dealing fire damage to anyone in the area.
tags predicted: [('evocation',)]
